RQ3 -----------------------------------------------

Random Forest

Dataset with dummy encoded state and covid 7 day rolling cases and deaths are considered here for the analysis.

In [21]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

yougov = pd.read_csv("yougov_oxcgrt_rolling_cleaned_preprocessed_withstate.csv") # dataset with dummy variables for state 

print(yougov.columns)
print(f"\nyougov columns:",len(yougov.columns))

Index(['RecordNo', 'Date', 'Non-household contacts', 'age', 'household_size',
       'Wellbeing', 'Perceived Severity', 'Perceived Susceptibility',
       'face_mask_scale', 'face_mask_binary',
       'general_protective_behavior_scale',
       'general_protective_behavior_binary',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_start_date', 'mandate_period',
       'Isolate if unwell_Not sure', 'Isolate if unwell_Yes',
       'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment'

Wave variable 

In [ ]:
# Waves duration

# Early Pandemic: Jan–Oct 2020
# Delta: Jun–Oct 2021
# Omicron: Nov 2021–Feb 2022


# create a yougov column
yougov["wave"] = "Not in wave" # place holder since the start


# Early Pandemic  
# set string 'Early Pandemic' for the wave column for the early pandemic period
yougov.loc[(yougov["Date"] >= "2020-01-01") & (yougov["Date"] <= "2020-10-31"),"wave"] = "Early Pandemic"

# Delta
# set string 'Delta' for the wave column for the Delta period
yougov.loc[(yougov["Date"] >= "2021-06-01") & (yougov["Date"] <= "2021-10-31"),"wave"] = "Delta"

# Omicron
yougov.loc[(yougov["Date"] >= "2021-11-01") & (yougov["Date"] <= "2022-02-28"),"wave"] = "Omicron"


yougov = yougov[yougov["wave"] != "Not in wave"] # keep only the data relevant to wave variable

print(yougov["wave"].value_counts())

wave
Delta             10787
Omicron            8629
Early Pandemic     8124
Name: count, dtype: int64


3 wave datasets

In [ ]:
# create 3 wave datasets
early = yougov[yougov["wave"] == "Early Pandemic"]  # early pandemic

delta = yougov[yougov["wave"] == "Delta"] # delta

omicron = yougov[yougov["wave"] == "Omicron"] # omicron



early.to_csv("wave_early_pandemic.csv", index=False)

delta.to_csv("wave_delta.csv", index=False)

omicron.to_csv("wave_omicron.csv", index=False)

In [24]:
print(early.isna().sum())
print(delta.isna().sum())
print(omicron.isna().sum())

RecordNo                                     0
Date                                         0
Non-household contacts                       0
age                                          0
household_size                               0
                                            ..
How Government handle situation_Very well    0
comorbidities_Prefer_not_to_say              0
comorbidities_Yes                            0
comorbidities_consent_removed                0
wave                                         0
Length: 69, dtype: int64
RecordNo                                     0
Date                                         0
Non-household contacts                       0
age                                          0
household_size                               0
                                            ..
How Government handle situation_Very well    0
comorbidities_Prefer_not_to_say              0
comorbidities_Yes                            0
comorbidities_consent_removed      

Data Splitting

Splitting Face mask wearing into early, delta, omicron waves

In [ ]:
# Data splitting 80-20%  - face mask 

from sklearn.model_selection import train_test_split



# early 

# 80% Train  20% Test 
early_train_facemask, early_test_facemask = train_test_split(
    early, test_size=0.20, random_state=42, stratify = early["face_mask_binary"]) # stratify by face mask binary

print("Early pandemic facemask:")
print("Training:", len(early_train_facemask))
print("Test:", len(early_test_facemask))


early_train_facemask.to_csv("early_train_facemask.csv", index=False)
early_test_facemask.to_csv("early_test_facemask.csv", index=False)




# delta 

# 80% Train  20% Test 
delta_train_facemask, delta_test_facemask = train_test_split(
    delta, test_size=0.20, random_state=42, stratify = delta["face_mask_binary"])

print("\nDelta facemask:")
print("Training:", len(delta_train_facemask))
print("Test:", len(delta_test_facemask))


delta_train_facemask.to_csv("delta_train_facemask.csv", index=False)
delta_test_facemask.to_csv("delta_test_facemask.csv", index=False)





# omicron 

# 80% Train  20% Test 
omicron_train_facemask, omicron_test_facemask = train_test_split(
    omicron, test_size=0.20, random_state=42, stratify = omicron["face_mask_binary"])

print("\nOmicron facemask:")
print("Training:", len(omicron_train_facemask))
print("Test:", len(omicron_test_facemask))


omicron_train_facemask.to_csv("omicron_train_facemask.csv", index=False)
omicron_test_facemask.to_csv("omicron_test_facemask.csv", index=False)


Early pandemic facemask:
Training: 6499
Test: 1625

Delta facemask:
Training: 8629
Test: 2158

Omicron facemask:
Training: 6903
Test: 1726


Check Target class

In [26]:
print("early train:", early_train_facemask["face_mask_binary"].value_counts(normalize=True))
print("early test:",early_test_facemask["face_mask_binary"].value_counts(normalize=True))

print("delta train:",delta_train_facemask["face_mask_binary"].value_counts(normalize=True))
print("delta test:",delta_test_facemask["face_mask_binary"].value_counts(normalize=True))

print("omicron train:",omicron_train_facemask["face_mask_binary"].value_counts(normalize=True))
print("omicron test:",omicron_test_facemask["face_mask_binary"].value_counts(normalize=True))

early train: face_mask_binary
0    0.674565
1    0.325435
Name: proportion, dtype: float64
early test: face_mask_binary
0    0.674462
1    0.325538
Name: proportion, dtype: float64
delta train: face_mask_binary
1    0.685247
0    0.314753
Name: proportion, dtype: float64
delta test: face_mask_binary
1    0.685357
0    0.314643
Name: proportion, dtype: float64
omicron train: face_mask_binary
1    0.782993
0    0.217007
Name: proportion, dtype: float64
omicron test: face_mask_binary
1    0.783314
0    0.216686
Name: proportion, dtype: float64


Splitting General protective behaviour into early, delta, omicron waves

In [ ]:
# Data splitting 80-20%  - General protective behaviour


# early 

# 80% Train  20% Test 
early_train_general_behav, early_test_general_behav = train_test_split(
    early, test_size=0.20, random_state=42, stratify = early["general_protective_behavior_binary"])

print("Early pandemic general_behav:")
print("Training:", len(early_train_general_behav))
print("Test:", len(early_test_general_behav))


early_train_general_behav.to_csv("early_train_general_behav.csv", index=False)
early_test_general_behav.to_csv("early_test_general_behav.csv", index=False)




# delta 

# 80% Train  20% Test 
delta_train_general_behav, delta_test_general_behav = train_test_split(
    delta, test_size=0.20, random_state=42, stratify = delta["general_protective_behavior_binary"])

print("\nDelta facemask:")
print("Training:", len(delta_train_general_behav))
print("Test:", len(delta_test_general_behav))


delta_train_general_behav.to_csv("delta_train_general_behav.csv", index=False)
delta_test_general_behav.to_csv("delta_test_general_behav.csv", index=False)





# omicron 

# 80% Train  20% Test 
omicron_train_general_behav, omicron_test_general_behav = train_test_split(
    omicron, test_size=0.20, random_state=42, stratify = omicron["general_protective_behavior_binary"])

print("\nOmicron facemask:")
print("Training:", len(omicron_train_general_behav))
print("Test:", len(omicron_test_general_behav))


omicron_train_general_behav.to_csv("omicron_train_general_behav.csv", index=False)
omicron_test_general_behav.to_csv("omicron_test_general_behav.csv", index=False)



Early pandemic general_behav:
Training: 6499
Test: 1625

Delta facemask:
Training: 8629
Test: 2158

Omicron facemask:
Training: 6903
Test: 1726


Check Target class

In [28]:
print("early train:", early_train_general_behav["general_protective_behavior_binary"].value_counts(normalize=True))
print("early test:",early_test_general_behav["general_protective_behavior_binary"].value_counts(normalize=True))

print("delta train:",delta_train_general_behav["general_protective_behavior_binary"].value_counts(normalize=True))
print("delta test:",delta_test_general_behav["general_protective_behavior_binary"].value_counts(normalize=True))

print("omicron train:",omicron_train_general_behav["general_protective_behavior_binary"].value_counts(normalize=True))
print("omicron test:",omicron_test_general_behav["general_protective_behavior_binary"].value_counts(normalize=True))

early train: general_protective_behavior_binary
1    0.628251
0    0.371749
Name: proportion, dtype: float64
early test: general_protective_behavior_binary
1    0.628308
0    0.371692
Name: proportion, dtype: float64
delta train: general_protective_behavior_binary
1    0.749218
0    0.250782
Name: proportion, dtype: float64
delta test: general_protective_behavior_binary
1    0.749305
0    0.250695
Name: proportion, dtype: float64
omicron train: general_protective_behavior_binary
1    0.717514
0    0.282486
Name: proportion, dtype: float64
omicron test: general_protective_behavior_binary
1    0.717845
0    0.282155
Name: proportion, dtype: float64
